# Beep detection helper
Detect initial beep-like pattern in a WAV file and report its end time.

In [8]:
import numpy as np
import soundfile as sf
from pathlib import Path


def detect_loud_to_silence(data, fs, loud_threshold=0.25, silence_threshold=0.05, smooth_ms=10, min_silence_ms=80):
    """Return the loud start time, next silence start time, and the interval between them."""
    if data.ndim > 1:
        data = data.mean(axis=1)

    envelope = np.abs(data)
    window = max(1, int(fs * (smooth_ms / 1000.0)))
    kernel = np.ones(window, dtype=float) / window
    envelope = np.convolve(envelope, kernel, mode='same')

    if envelope.max() > 0:
        envelope = envelope / envelope.max()

    loud = envelope >= loud_threshold
    if not np.any(loud):
        return None, None, None

    loud_start = np.argmax(loud)
    silence_samples = int(fs * (min_silence_ms / 1000.0))
    silence_start = None
    i = loud_start
    while i < len(envelope):
        if envelope[i] <= silence_threshold:
            j = i
            while j < len(envelope) and envelope[j] <= silence_threshold:
                j += 1
            if (j - i) >= silence_samples:
                silence_start = i
                break
            i = j
        else:
            i += 1

    if silence_start is None:
        return loud_start / fs, None, None

    loud_time = loud_start / fs
    silence_time = silence_start / fs
    return loud_time, silence_time, silence_time - loud_time


def detect_loud_to_silence_file(file_path, **kwargs):
    file_path = Path(file_path)
    data, fs = sf.read(file_path)
    loud_time, silence_time, interval = detect_loud_to_silence(data, fs, **kwargs)

    print(f'File: {file_path}')
    if loud_time is None:
        print('  No loud sound detected.')
    elif silence_time is None:
        print(f'  Loud sound starts at {loud_time:.4f}s but no silence was reached afterward.')
    else:
        print(f'  Loud sound starts at {loud_time:.4f}s')
        print(f'  Next silence begins at {silence_time:.4f}s')
        print(f'  Time between loud sound and next silence: {interval:.4f}s')
    return loud_time, silence_time, interval


for filename in ['r_1.wav', 'r_2.wav', 'r_4.wav']:
    wav_file = Path('..') / 'Recordings' / 'User0' / filename
    detect_loud_to_silence_file(wav_file)


File: ..\Recordings\User0\r_1.wav
  Loud sound starts at 4.4357s
  Next silence begins at 5.6728s
  Time between loud sound and next silence: 1.2371s
File: ..\Recordings\User0\r_2.wav
  Loud sound starts at 3.2949s
  Next silence begins at 4.5350s
  Time between loud sound and next silence: 1.2401s
File: ..\Recordings\User0\r_4.wav
  Loud sound starts at 3.3729s
  Next silence begins at 4.5031s
  Time between loud sound and next silence: 1.1302s
